In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.linalg import solve_triangular

In [2]:
fh = "prostate.csv"
df = pd.read_csv(fh)

df.sample(n=3)

,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa
41,1.442202,3.6826,68,-1.386294,0,-1.38629,7,10,2.30757
73,1.838961,3.2367,60,0.438255,1,1.17865,9,90,3.07501
51,1.660131,4.2348,64,2.073172,0,-1.38629,6,0,2.67759


### Fitting w/ NumPy

In [3]:
# Define response and predictors

x_cols = df.columns.drop('lpsa').to_list()

y = df.loc[:, 'lpsa'].values
X = df.loc[:, x_cols].values

X[:3, :]

array([[-0.5798185,  2.7695   , 50.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.9942523,  3.3196   , 58.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.5108256,  2.6912   , 74.       , -1.386294 ,  0.       ,
        -1.38629  ,  7.       , 20.       ]])

In [4]:
X.shape

(97, 8)

In [5]:
# Add an intercept

X = np.hstack(
    [np.ones(shape=(X.shape[0], 1)), X]
)

X[:3, :]

array([[ 1.       , -0.5798185,  2.7695   , 50.       , -1.386294 ,
         0.       , -1.38629  ,  6.       ,  0.       ],
       [ 1.       , -0.9942523,  3.3196   , 58.       , -1.386294 ,
         0.       , -1.38629  ,  6.       ,  0.       ],
       [ 1.       , -0.5108256,  2.6912   , 74.       , -1.386294 ,
         0.       , -1.38629  ,  7.       , 20.       ]])

In [6]:
# Use NumPy's OLS function

beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)

beta

array([ 0.6693367 ,  0.58702183,  0.45446742, -0.01963718,  0.10705403,
        0.76615733, -0.10547426,  0.0451416 ,  0.00452523])

In [7]:
# Coeffiecients

x_cols_w_int = ['intercept'] + x_cols

res = pd.DataFrame(
    data = {c: [b] for c, b in zip(x_cols_w_int, beta)},
    index = ['coef']
)

res

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525


### Fitting w/ Algorithm 3.1

In [8]:
# Define algorithm 3.1

def algorithm_3_1_all_coef(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Returns all coefficients in a computationally efficient way."""
    n, p = X.shape
    Z = np.empty(shape=(n, p + 1))
    Z[:, 0] = 1.0
    D = np.zeros(shape=(p + 1, p + 1))      # X = Z D^{-1} D G
    G = np.eye(p + 1)
    D[0,0] = np.sqrt(n)

    for j in range(1, p + 1):
        z_j = X[:, j - 1].astype(float)

        for l in range(j):
            z_l   = Z[:, l]
            gamma = np.dot(z_j, z_l) / np.dot(z_l, z_l)
            G[l, j] = gamma
            z_j -= gamma * z_l

        z_norm = np.linalg.norm(z_j)
        Z[:, j] = z_j
        D[j,j] = z_norm

    Q = Z / np.diag(D)
    R = D @ G

    return solve_triangular(R, Q.T@y, lower=False)

In [9]:
# Define response and predictors

y = df.loc[:, 'lpsa'].values
X = df.loc[:, x_cols].values

X[:3, :]

array([[-0.5798185,  2.7695   , 50.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.9942523,  3.3196   , 58.       , -1.386294 ,  0.       ,
        -1.38629  ,  6.       ,  0.       ],
       [-0.5108256,  2.6912   , 74.       , -1.386294 ,  0.       ,
        -1.38629  ,  7.       , 20.       ]])

In [10]:
# All Coeffiecients

coefs = algorithm_3_1_all_coef(X, y)

a = pd.DataFrame(
    data = {c: [b] for c, b in zip(x_cols_w_int, coefs)},
    index = ['coef']
)

a

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525


### Fitting w/ Statsmodels (extra)

In [11]:
X = df.drop(columns='lpsa')
y = df.loc[:, 'lpsa']

X = sm.add_constant(X)
model = sm.OLS(y, X).fit()

params = model.params

params.name = 'coef'
params = params.rename(index={'const': 'intercept'})

res = pd.DataFrame(
    data = {c: [b] for c, b in params.items()},
    index = ['coef']
)

res

,intercept,lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45
coef,0.669337,0.587022,0.454467,-0.019637,0.107054,0.766157,-0.105474,0.045142,0.004525


In [12]:
a.round(6).equals(res.round(6))

True